# 第3回 分類の評価 — 実習ノートブック

**このノートブックは第3回だけで完結します。** 必要なのは割り算だけです。

やること:
1. 正解率99.9%なのに患者を1人も見つけない検査を作る
2. 混同行列・再現率・適合率を計算する
3. 閾値を動かして、2つの指標が逆向きに動くのを見る
4. 同じ検査でも使う場所で意味が変わることを確かめる

所要時間:実習 約20分 / 課題 約40分

## 0. 準備

In [ ]:
# 準備(このセルを最初に1回だけ実行)
!pip install -q japanize-matplotlib
import numpy as np, matplotlib.pyplot as plt, japanize_matplotlib
plt.rcParams["font.size"] = 12
乱数 = np.random.default_rng(3)
print("準備できました")

## 1. まれな病気の検査を作る

1000人に1人がかかる病気を考えます。1万人を検査する状況をシミュレーションします。

In [ ]:
# ===== 1万人に検査をする状況を作る =====
人数 = 10000
有病率 = 0.001          # 1000人に1人

実際に患者 = 乱数.random(人数) < 有病率
print(f"{人数}人のうち、実際の患者は {実際に患者.sum()}人({実際に患者.mean():.2%})")

## 2. 2つの検査を比べる

- ① **「全員、陰性です」と答えるだけ**の検査
- ② ふつうの検査(感度90%・特異度95%)

正解率だけを見ると、どちらが勝つでしょうか。

In [ ]:
# ===== 指標を計算する関数(これだけ用意すればあとは使い回せる)=====
def 評価(実際, 判定, 名=""):
    真陽性 = int(( 実際 &  判定).sum())   # 患者を正しく陽性と判定
    偽陽性 = int((~実際 &  判定).sum())   # 健康な人を陽性と判定 = 誤検出
    偽陰性 = int(( 実際 & ~判定).sum())   # 患者を陰性と判定 = 見逃し
    真陰性 = int((~実際 & ~判定).sum())   # 健康な人を正しく陰性と判定
    正解率 = (真陽性 + 真陰性) / len(実際)
    再現率 = 真陽性 / (真陽性 + 偽陰性) if (真陽性 + 偽陰性) else float("nan")
    適合率 = 真陽性 / (真陽性 + 偽陽性) if (真陽性 + 偽陽性) else float("nan")
    if 名:
        print(f"【{名}】")
        print(f"            実際は陰性   実際は陽性")
        print(f"  陰性と判定  {真陰性:>8}   {偽陰性:>8}  ← 右は「見逃し」")
        print(f"  陽性と判定  {偽陽性:>8}   {真陽性:>8}  ← 左は「誤検出」")
        print(f"  正解率 {正解率:.2%} / 再現率 {再現率:.1%} / 適合率 "
              + (f"{適合率:.1%}" if 適合率 == 適合率 else "計算できない(陽性と1件も言っていない)"))
        print()
    return dict(正解率=正解率, 再現率=再現率, 適合率=適合率,
                真陽性=真陽性, 偽陽性=偽陽性, 偽陰性=偽陰性, 真陰性=真陰性)

# ① 「全員、陰性です」と答える検査
全員陰性 = np.zeros(人数, dtype=bool)
r1 = 評価(実際に患者, 全員陰性, "① 全員を陰性と答える検査")

# ② ふつうの検査(感度90% = 患者の9割を陽性にできる / 特異度95% = 健康な人の5%を誤検出)
判定2 = np.where(実際に患者, 乱数.random(人数) < 0.90, 乱数.random(人数) < 0.05)
r2 = 評価(実際に患者, 判定2, "② ふつうの検査(感度90%・特異度95%)")

print("正解率は①の勝ち。しかし患者を見つけるのは②だけです。")

## 3. 閾値というつまみ

モデルは本当は「陽性/陰性」ではなく、**0〜1の「陽性らしさ」** を出しています。
どこに線を引くか(閾値)は、あとから自由に決められます。

In [ ]:
# ===== 本当は「陽性らしさ」という数字が出ている =====
# モデルは 0〜1 の「陽性らしさ」を出し、どこかに線(閾値)を引いて陽性/陰性を決めています。
陽性らしさ = np.where(実際に患者,
                      乱数.beta(5, 2, 人数),      # 患者は高めの値が出やすい
                      乱数.beta(2, 8, 人数))      # 健康な人は低めの値が出やすい

plt.figure(figsize=(8, 4))
plt.hist(陽性らしさ[~実際に患者], bins=40, alpha=.7, label="健康な人", color="#3A46A8")
plt.hist(陽性らしさ[実際に患者], bins=40, alpha=.9, label="患者", color="#E08A2B")
plt.axvline(0.5, color="#C0392B", ls="--", lw=2, label="閾値 0.5")
plt.yscale("log")     # 患者が少なすぎるので対数目盛にしています
plt.xlabel("モデルが出した「陽性らしさ」"); plt.ylabel("人数(対数)")
plt.title("閾値をどこに引くかは、あとから決められる")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

print("2つの山は重なっています。どこで切っても、どちらかの間違いが必ず残ります。")

In [ ]:
# ===== 演習・問2:閾値を動かす =====
print(f"{'閾値':>6}{'陽性と判定':>12}{'再現率':>10}{'適合率':>10}{'正解率':>10}")
結果 = []
for 閾値 in [0.1, 0.3, 0.5, 0.7, 0.9]:
    r = 評価(実際に患者, 陽性らしさ >= 閾値)
    結果.append((閾値, r))
    print(f"{閾値:>6}{r['真陽性']+r['偽陽性']:>12}{r['再現率']:>9.1%}{r['適合率']:>10.1%}{r['正解率']:>10.2%}")

plt.figure(figsize=(7.5, 4))
閾値list = [z[0] for z in 結果]
plt.plot(閾値list, [z[1]['再現率']*100 for z in 結果], "o-", color="#3A46A8", lw=2.5, label="再現率(取りこぼさなさ)")
plt.plot(閾値list, [z[1]['適合率']*100 for z in 結果], "s-", color="#E08A2B", lw=2.5, label="適合率(当たりの割合)")
plt.xlabel("閾値"); plt.ylabel("%"); plt.title("閾値を上げると、再現率は下がり適合率は上がる")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

print("2本の線が逆向きに動きます。両方を同時に上げることはできません。")

## 4. 同じ検査でも、使う場所で意味が変わる(演習・問3)

**検査の性能は一切変えません。** 変えるのは「対象集団に患者がどれだけいるか」だけです。

In [ ]:
# ===== 演習・問3:病気の割合を変える(検査の性能は同じまま)=====
print("検査の性能(感度90%・特異度95%)は一切変えていません。変えるのは有病率だけです。\n")
print(f"{'有病率':>8}{'患者数':>9}{'再現率':>10}{'適合率':>10}  陽性と言われたときの意味")
for p in [0.001, 0.01, 0.05, 0.10, 0.30]:
    g = np.random.default_rng(3)
    患者 = g.random(人数) < p
    判定 = np.where(患者, g.random(人数) < 0.90, g.random(人数) < 0.05)
    r = 評価(患者, 判定)
    印 = "ほぼ外れ" if r['適合率'] < 0.1 else ("半々くらい" if r['適合率'] < 0.6 else "かなり確か")
    print(f"{p:>7.1%}{患者.sum():>9}{r['再現率']:>10.1%}{r['適合率']:>10.1%}  {印}")

print("\n同じ検査なのに、適合率だけが大きく変わります。")
print("理由:適合率の分母には「健康な人の誤検出」が入るため。健康な人が多いほど誤検出の絶対数が増え、本物が埋もれます。")
print("→ 同じAIでも、使う場所によって「陽性」の重みが変わります。")

## 5. 自分で試す

In [ ]:
# ===== 自分で試す =====
# 感度・特異度・有病率を書きかえて、指標がどう動くか確かめてください。
感度   = 0.90     # 患者を陽性と判定できる割合
特異度 = 0.95     # 健康な人を陰性と判定できる割合
有病率2 = 0.001

g = np.random.default_rng(0)
患者 = g.random(人数) < 有病率2
判定 = np.where(患者, g.random(人数) < 感度, g.random(人数) < (1 - 特異度))
評価(患者, 判定, f"感度{感度:.0%}・特異度{特異度:.0%}・有病率{有病率2:.1%}")

print("試してみてください:特異度を 0.95 → 0.999 に上げると、適合率はどうなりますか?")

## 6. 提出課題(manaba)

**締切:次回講義の前日 23:59**

### 課題1:身のまわりの「まれなものを見つける」場面を分析する
病気の検査 **以外** から1つ選んでください。

- (a) 何を見つけますか
- (b) **見逃し**(偽陰性)が起きると、**誰が**どう困りますか
- (c) **誤検出**(偽陽性)が起きると、**誰が**どう困りますか
- (d) どちらの間違いをより避けるべきですか。理由も
- (e) 閾値は高め・低めのどちらにしますか

> (b)(c) は「誰が」まで書いてください。困る人が違うことがあります。
> 例:採用の書類選考では、見逃しで困るのは会社、誤検出で困るのは応募者です。

### 課題2:ノートブックを動かして報告する
- (a) 閾値を変えたときの再現率・適合率の表とグラフ
- (b) 有病率を変えたときの適合率の表
- (c) (b) の結果を自分の言葉で3行以上で説明する

### 課題3:次の主張を批判する(300字以上)

> 「当社のAIによる不正取引検知システムは、精度99.5%を達成しました。
> 従来の人手による確認は95%でしたので、大幅な改善です。今後は本システムによる自動判定に切り替えます。」

**ヒント:不正取引は全体の何%くらいだと思いますか。** そこから考え始めてください。

考える材料:
- その精度は、何もしないシステムと比べてどうですか
- 人手の95%と、同じ指標・同じ対象で比べていますか
- 「自動判定に切り替える」で、誤検出された人はどうなりますか
- 間違えられた人は、どこに申し立てればよいのですか

**加点**:「どういう数字が示されれば納得するか」まで書けると評価が上がります。